# SymFormer / TBX11K — Colab Runbook (Core-Method PoC)

Executes the phases in `plan.md` on a free Colab **T4**. Run cells top-to-bottom.

**Storage split** (free Drive is only 15GB; the Colab VM has ~100GB of ephemeral disk):

| What | Where | Why |
|---|---|---|
| code / configs / docs | **GitHub** (cloned to `/content/FOP`) | tiny, versioned |
| raw TBX11K (tens of GB) | **`/content`** (ephemeral) | download → prep → discard |
| compact TB-only 512² dataset (~few hundred MB) | **Drive** | expensive to rebuild |
| checkpoints + logs | **Drive** | needed to resume after a time-out |

Training cells use `--resume`, so after a session time-out just re-run the same cell.
See `paper.md` (method + target numbers), `CLAUDE.md` (scope/recipe), `limitations.md` (caveats).

## Phase 1 — environment

In [ ]:
!nvidia-smi   # confirm a GPU (ideally a T4) is attached; if none, change runtime or retry later

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# ---- paths ----
# Storage split: free Drive is only 15GB, but the Colab VM has ~100GB of EPHEMERAL local disk.
#   raw TBX11K (tens of GB) -> /content  (download, prep, then let it die with the session)
#   compact 512 dataset + checkpoints/logs -> Drive (must survive time-outs)
REPO_URL = ''                             # <-- your GitHub repo URL (used by the next cell)
PROJECT  = '/content/FOP'                 # this repo on Colab
DRIVE    = '/content/drive/MyDrive'
DATA_RAW = '/content/TBX11K'              # raw TBX11K — EPHEMERAL, deliberately NOT on Drive
DATA_512 = f'{DRIVE}/tbx11k_tb512'        # compact TB-only 512 dataset (built in Phase 3)
RUNS     = f'{DRIVE}/tb_runs'             # checkpoints + logs (persist across sessions)
os.makedirs(RUNS, exist_ok=True)
print('project       :', PROJECT)
print('raw (temp)    :', DATA_RAW)
print('compact(Drive):', DATA_512)
print('runs (Drive)  :', RUNS)

In [ ]:
# Pull the code from GitHub (set REPO_URL above). Re-running this in a later session picks up
# anything you've pushed since.
assert REPO_URL, 'Set REPO_URL in the cell above first.'
if os.path.isdir(PROJECT):
    !cd {PROJECT} && git pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT}
%cd {PROJECT}
!ls

In [ ]:
# Install a compatible mm stack via mim (it resolves mmcv for the installed torch/CUDA).
!pip install -q -U openmim
!mim install -q mmengine
!mim install -q "mmcv>=2.0.0,<2.2.0"
!mim install -q "mmdet>=3.0.0,<3.4.0"
!pip install -q pycocotools

In [ ]:
# Verify + freeze the exact versions (RECORD these in results.md).
import torch, mmcv, mmdet, mmengine
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('mmcv', mmcv.__version__, 'mmdet', mmdet.__version__, 'mmengine', mmengine.__version__)
open('requirements.lock.txt','w').write(
    f"torch=={torch.__version__}\nmmcv=={mmcv.__version__}\n"
    f"mmdet=={mmdet.__version__}\nmmengine=={mmengine.__version__}\n")
print('wrote requirements.lock.txt')

## Phase 6 (early) — unit tests
Run the SAS tests now that torch is available; they gate the model code.

In [ ]:
!python tests/test_numpy_ref.py
!python tests/test_sas.py

## Phases 2–3 — data
Download the raw TBX11K to **ephemeral `/content`** (NOT Drive), build the compact TB-only 512²
set, and write only that to Drive. After this, later sessions need only the compact set.

In [ ]:
# sanity-check the prep logic first (synthetic, no real data):
!python tools/prepare_tbx11k.py --selftest

In [ ]:
# Skip this whole cell if {DATA_512} already exists on Drive from a previous session.
import os
if os.path.isdir(DATA_512):
    print('compact dataset already on Drive ->', DATA_512, '(skip the download/prep)')
else:
    os.makedirs(DATA_RAW, exist_ok=True)
    print('Download TBX11K into', DATA_RAW, 'using the links at')
    print('  https://github.com/yun-liu/Tuberculosis')
    print('e.g.  !gdown <FILE_ID> -O /content/tbx11k.zip  &&  !unzip -q /content/tbx11k.zip -d /content')
    print('Then run the next cell. Raw data stays on /content and is discarded with the session.')
!df -h /content | tail -1   # check free ephemeral disk

In [ ]:
# Build the compact TB-only 512 COCO dataset -> Drive. Adjust --xml-dir/--img-dir/--*-list to
# match your TBX11K copy's layout. Omit the list flags to fall back to a deterministic 3:1 split.
!python tools/prepare_tbx11k.py --src {DATA_RAW} --dst {DATA_512} \
    --xml-dir annotations/xml --img-dir imgs \
    --train-list {DATA_RAW}/lists/TB_train.txt --val-list {DATA_RAW}/lists/TB_val.txt \
    --size 512 --write-agnostic
!du -sh {DATA_512}   # expect a few hundred MB

## Phase 4 — smoke test (1 epoch, tiny batch)

In [ ]:
!python tools/train_runner.py configs/retinanet_r50_fpn_tbx11k_512.py \
    --work-dir {RUNS}/smoke --data-root {DATA_512}/ --max-epochs 1 --batch-size 2

## Phase 5 — RetinaNet baseline (re-run this cell after any time-out; it resumes)

In [ ]:
!python tools/train_runner.py configs/retinanet_r50_fpn_tbx11k_512.py \
    --work-dir {RUNS}/baseline --data-root {DATA_512}/ --resume

In [ ]:
!python tools/test_runner.py configs/retinanet_r50_fpn_tbx11k_512.py \
    {RUNS}/baseline/epoch_24.pth --data-root {DATA_512}/

## Phase 7 — SymFormer w/ RetinaNet (the primary comparison)

In [ ]:
!python tools/train_runner.py configs/symformer_retinanet_r50_fpn_tbx11k_512.py \
    --work-dir {RUNS}/symformer --data-root {DATA_512}/ --resume

In [ ]:
!python tools/test_runner.py configs/symformer_retinanet_r50_fpn_tbx11k_512.py \
    {RUNS}/symformer/epoch_24.pth --data-root {DATA_512}/
# GATE: SymFormer AP50 should exceed the baseline AP50. Record both in results.md.

## Phase 8 — Table 8 ablation (long; each cell resumes independently)
Checkpoints are deleted after each cell is evaluated so Drive usage stays bounded
(~300MB per checkpoint x 14 runs would otherwise exceed the 15GB free tier).

In [ ]:
import glob, subprocess, os, shutil, json
DELETE_CKPTS_AFTER_EVAL = True   # set False if you want to keep every ablation checkpoint
# baseline first, then the most-informative cells (see configs/ablation/README.md)
order = [
    'configs/retinanet_r50_fpn_tbx11k_512.py',
    'configs/ablation/abl_vanilla_ape.py',
    'configs/ablation/abl_symattention_ape.py',
    'configs/ablation/abl_symattention_spe_stn_r2l.py',
    'configs/ablation/abl_symattention_spe_nostn_r2l.py',
    'configs/ablation/abl_symattention_spe_stn_l2r.py',
]
rest = sorted(set(glob.glob('configs/ablation/abl_*.py')) - set(order))
for c in order + rest:
    name = os.path.splitext(os.path.basename(c))[0]
    wd = f'{RUNS}/abl/{name}'
    print('==== TRAIN', name, '=========================')
    subprocess.run(['python','tools/train_runner.py',c,'--work-dir',wd,
                    '--data-root',f'{DATA_512}/','--resume'])
    print('==== EVAL', name, '=========================')
    subprocess.run(['python','tools/test_runner.py',c,f'{wd}/epoch_24.pth',
                    '--data-root',f'{DATA_512}/'])
    if DELETE_CKPTS_AFTER_EVAL:
        for p in glob.glob(f'{wd}/*.pth'):
            os.remove(p)   # keep the logs/metrics, drop the ~300MB weights
        print('removed checkpoints for', name)
!du -sh {RUNS}

## Phase 10 — write-up
Fill `results.md` (baseline vs SymFormer, and the full ablation table), update `CLAUDE.md` §0/§6/§8
with the actual numbers, and note deviations (batch size, versions, filled-in defaults) and caveats
(val-not-test; reduced data → trend not absolute numbers). Then commit and push `results.md`.